# Interface Plates — Structural Theory

Validates each plate variant against the two load cases that govern its duty cycle:

1. **Bolt pattern under fault current** — joint heating during 5-cycle bolted-fault clear; joint must stay below 6061-T6 yield-derating threshold (150°C).
2. **Thermal expansion differential** — plate (6061 alpha=23.6e-6/K) vs receiver frame (A36 alpha=11.7e-6/K); per-corner-bolt radial offset must stay within bolt clearance.

Per-plate inputs in `sim/{plate_id}/constants.py`. Solver in `sim/{plate_id}/model.py`. Assertions in `sim/{plate_id}/test_run.py`.

Other load cases (deflection, stress concentration at penetrations, wind load) are bounded by inspection — see ADR-017 in `edp-module-assemblies/system_adrs.md`.

## CD plate — coolant flow & QD body OD derivation

CD is the only plate where conduit OD is **derived** rather than dictated by an upstream cable bundle. Two coolant lines (supply + return) carry the secondary loop between compute container CDU and external drycooler. QD body OD drives `power_conduit_od_mm` in `cad/specs/CD/spec.yaml`.

This is upstream of the structural loop below: it determines penetration diameter, which is then fed into the same fault-current + thermal-expansion analysis as every other plate.

### Assumptions

- **Heat load to reject**: 70 kW ± 5 kW per compute container (7× SYS-421GE-NBRT-LCC nodes × ~10 kW liquid load each: 8× HGX B200 @ 1000 W TDP + CPU/DIMM/NIC cold plates, 95% to liquid). Source: NVIDIA HGX B200 product brief; Lenovo HGX B200 1000W product guide.
- **Working fluid**: deionized water, single-phase (no PGW). cp = 4180 J/(kg·K), ρ = 1000 kg/m³.
- **Secondary loop ΔT**: 5 K ± 1 K (drycooler design point at 32 °C ambient with 5 K approach).
- **Topology**: 2 parallel hydraulic lines (supply + return), each carrying full mass flow (not split — one direction only).
- **QD candidate**: Stäubli SBX 50 (2″ class) OR equivalent (Parker QCM, CPC LQ8). Hub OD ≈ 75 mm, rated 250 LPM at <0.5 bar pressure drop.
- **Neglected**: pipe friction between CDU and CD plate (CDU pump compensates), localized turbulence at QD body (<2% of total ΔP), seasonal density variation (<0.3% over 0–40 °C).
- **Out of scope at v1**: redundant N+1 cooling line; coolant freeze protection (commercial deployments only, > 0 °C).

In [1]:
import pint
from uncertainties import ufloat

ureg = pint.UnitRegistry()

# --- Inputs (with uncertainty per assumptions cell above) ---
HEAT_LOAD = ufloat(70, 5) * ureg.kW            # 7 nodes x ~10 kW liquid load
WATER_CP = 4180.0 * ureg.J / (ureg.kg * ureg.kelvin)
WATER_DENSITY = 1000.0 * ureg.kg / ureg.m**3
DELTA_T_SECONDARY = ufloat(5, 1) * ureg.kelvin  # drycooler design point

# --- Mass flow: m_dot = Q / (cp * dT) ---
m_dot = (HEAT_LOAD / (WATER_CP * DELTA_T_SECONDARY)).to(ureg.kg / ureg.s)

# --- Volumetric flow: V_dot = m_dot / rho ---
v_dot = (m_dot / WATER_DENSITY).to(ureg.liter / ureg.minute)

# Reason: each line (supply, return) carries the FULL mass flow — they are in
# series in the loop, not parallel branches splitting it.
v_dot_per_line = v_dot

print(f"Mass flow      : {m_dot:~P}")
print(f"Total volflow  : {v_dot:~P}")
print(f"Per line       : {v_dot_per_line:~P}")


Mass flow      : 3.3+/-0.7 kg/s
Total volflow  : (2.0+/-0.4)e+02 l/min
Per line       : (2.0+/-0.4)e+02 l/min


In [2]:
# --- Expected values (feed into sim/cd/constants.py) ---
# Pin the nominal so the sim assertion has a fixed target; tolerance comes from
# uncertainty bands above (±15% on flow rate, dominated by ΔT uncertainty).
EXPECTED_FLOW_PER_LINE_LPM = 200.0      # nominal, ±15%
EXPECTED_QD_BODY_OD_MM = 75.0            # Stäubli SBX 50 / Parker QCM 2"
EXPECTED_FLOW_REL_TOL = 0.15

# --- Sanity 1: order-of-magnitude vs published references ---
# GB200 NVL72 (72 GPUs total): >700 LPM container-internal primary loop.
# Our v1 secondary loop (per-container, 7x8=56 GPUs equivalent thermal):
# expected ~200 LPM is consistent with secondary being ~1/3 the primary
# rate due to higher dT (5 K vs 1 K).
ratio_to_nvl72 = v_dot.magnitude.nominal_value / 700.0
print(f"v1 CD flow / GB200 NVL72 primary : {ratio_to_nvl72:.2f} (expect ~0.3, secondary loop)")

# --- Sanity 2: QD pressure drop margin ---
# Stäubli SBX 50 rated 250+ LPM at <0.5 bar. At nominal 200 LPM:
qd_rated_flow_lpm = 250.0
flow_fraction = v_dot.magnitude.nominal_value / qd_rated_flow_lpm
print(f"Flow / SBX 50 rated capacity     : {flow_fraction:.2f} (must be < 1.0)")

# --- Sanity 3: line velocity (rule of thumb < 3 m/s for steel pipe) ---
qd_inner_dia = 50.0 * ureg.mm  # 2" nominal -> ~50mm bore
qd_area = (3.14159 * (qd_inner_dia / 2) ** 2).to(ureg.m**2)
v_line = (m_dot.magnitude.nominal_value * ureg.kg / ureg.s / WATER_DENSITY / qd_area).to(ureg.m / ureg.s)
print(f"Line velocity at nominal flow    : {v_line:~P.2f} (rule of thumb < 3 m/s)")

print()
print(f"=> EXPECTED_FLOW_PER_LINE_LPM = {EXPECTED_FLOW_PER_LINE_LPM} ± {EXPECTED_FLOW_REL_TOL*100:.0f}%")
print(f"=> EXPECTED_QD_BODY_OD_MM     = {EXPECTED_QD_BODY_OD_MM} (Stäubli SBX 50 class)")


v1 CD flow / GB200 NVL72 primary : 0.29 (expect ~0.3, secondary loop)
Flow / SBX 50 rated capacity     : 0.80 (must be < 1.0)
Line velocity at nominal flow    : 1.71 m/s (rule of thumb < 3 m/s)

=> EXPECTED_FLOW_PER_LINE_LPM = 200.0 ± 15%
=> EXPECTED_QD_BODY_OD_MM     = 75.0 (Stäubli SBX 50 class)


## Run all plates

In [3]:
import importlib

PLATES = ["cg", "bg_ac", "bg_dc", "cd"]
results = {}
for plate in PLATES:
    constants = importlib.import_module(f"sim.{plate}.constants")
    model = importlib.import_module(f"sim.{plate}.model")
    res = model.solve()
    results[plate] = (constants, res)
    print(f"=== {plate.upper()} ===")
    print(f"  joint temp rise  : {res.joint_temp_rise.to(constants.ureg.kelvin):.2f}")
    print(f"  thermal offset   : {res.thermal_offset.to(constants.ureg.mm):.3f}")
    print()


=== CG ===
  joint temp rise  : 50.80 kelvin
  thermal offset   : 0.449 millimeter



=== BG_AC ===
  joint temp rise  : 2.78 kelvin
  thermal offset   : 0.449 millimeter



=== BG_DC ===
  joint temp rise  : 3.21 kelvin
  thermal offset   : 0.449 millimeter



=== CD ===
  joint temp rise  : 0.09 kelvin
  thermal offset   : 0.449 millimeter



## Verdicts table

In [4]:
print(f"{'plate':<6} {'fault rise (K)':<18} {'fault verdict':<14} {'thermal off (mm)':<18} {'thermal margin (mm)':<22} {'thermal verdict':<14}")
print("-" * 100)
for plate_name, (consts, res) in results.items():
    rise_k = res.joint_temp_rise.to(consts.ureg.kelvin).magnitude
    fault_verdict = "PASS" if rise_k < (consts.JOINT_TEMP_THRESHOLD_C - consts.T_AMBIENT_FAULT_C) else "FAIL"

    offset_mm = res.thermal_offset.to(consts.ureg.mm).magnitude
    clearance_mm = consts.BOLT_CLEARANCE_RADIAL.to(consts.ureg.mm).magnitude
    margin_mm = clearance_mm - offset_mm
    thermal_verdict = "PASS" if margin_mm > 0 else "FAIL"

    print(f"{plate_name:<6} {rise_k:<18.2f} {fault_verdict:<14} {offset_mm:<18.3f} {margin_mm:<22.3f} {thermal_verdict:<14}")


plate  fault rise (K)     fault verdict  thermal off (mm)   thermal margin (mm)    thermal verdict
----------------------------------------------------------------------------------------------------
cg     50.80              PASS           0.449              0.051                  PASS          
bg_ac  2.78               PASS           0.449              0.051                  PASS          
bg_dc  3.21               PASS           0.449              0.051                  PASS          
cd     0.09               PASS           0.449              0.051                  PASS          


## Design risk mitigation — corner-slot adoption

Round corner-bolt holes leave a ~0.05 mm thermal margin (per the verdicts table above), which ISO 2768-m fab tolerance (±0.1 mm) consumes entirely. **Adopted mitigation: radially-slotted corner bolt holes.** Edge-midpoint bolts stay round (their radial offset is much smaller — they sit on the symmetry axes, not the diagonal).

Slot length derivation:

$$L_{slot} = D_{hole} + 2 \cdot (\delta_{thermal} + \delta_{fab} + \delta_{margin})$$

where $\delta_{thermal}$ is per-corner radial offset, $\delta_{fab}$ is hole-position tolerance per ISO 2768-m, and $\delta_{margin}$ is the engineering safety factor.

Slot orientation: each corner slot points radially toward the bolt-pattern center. The bolt rides axially (clamped via washer); slot absorbs radial growth/shrink.

In [5]:
# --- Slot length derivation (commercial CG geometry, applies to all plates) ---
# Same Δα + diagonal + ΔT across the 4-plate fleet → same slot length.
HOLE_DIAMETER_MM = 11.0          # M10 clearance hole
THERMAL_OFFSET_MM = 0.449        # per-corner radial offset at full ΔT (verdicts table)
FAB_TOLERANCE_MM = 0.1           # ISO 2768-m hole position
SAFETY_MARGIN_MM = 0.2           # engineering buffer

slot_length_mm = HOLE_DIAMETER_MM + 2 * (THERMAL_OFFSET_MM + FAB_TOLERANCE_MM + SAFETY_MARGIN_MM)
print(f"Computed slot length        : {slot_length_mm:.2f} mm")
print("Fab-spec slot length        : 13.00 mm  (rounded up)")

# Sanity: slot should accommodate full ±thermal offset + fab tol + margin in radial direction.
slot_radial_extra = (13.0 - HOLE_DIAMETER_MM) / 2  # mm beyond hole center on each side
budget_each_side = THERMAL_OFFSET_MM + FAB_TOLERANCE_MM + SAFETY_MARGIN_MM
print(f"Slot radial extra per side  : {slot_radial_extra:.2f} mm")
print(f"Budget per side             : {budget_each_side:.2f} mm  (must be ≤ slot extra)")
assert slot_radial_extra >= budget_each_side, "13 mm slot insufficient — recompute"

# Defense-extreme sanity: ΔT = 111 K (-40 to +71 °C MIL-STD-810H), what's the offset?
defense_offset_mm = THERMAL_OFFSET_MM * 111 / 85
defense_budget = defense_offset_mm + FAB_TOLERANCE_MM + SAFETY_MARGIN_MM
print(f"Defense ΔT=111K offset      : {defense_offset_mm:.3f} mm; budget: {defense_budget:.2f} mm")
print(f"Defense headroom in 13 mm   : {slot_radial_extra - defense_budget:.2f} mm  (must be ≥ 0)")


Computed slot length        : 12.50 mm
Fab-spec slot length        : 13.00 mm  (rounded up)
Slot radial extra per side  : 1.00 mm
Budget per side             : 0.75 mm  (must be ≤ slot extra)
Defense ΔT=111K offset      : 0.586 mm; budget: 0.89 mm
Defense headroom in 13 mm   : 0.11 mm  (must be ≥ 0)


In [6]:
# Sanity 1: fault energy <<< BESS capacity. Should be ~1e-7 ratio.
for plate_name, (consts, _res) in results.items():
    n_eff = consts.BOLT_COUNT / 2
    i_worst = (consts.FAULT_CURRENT.magnitude.nominal_value + consts.FAULT_CURRENT.magnitude.std_dev) * consts.ureg.kA
    r_worst = (consts.R_JOINT_PER_BOLT.magnitude.nominal_value + 2 * consts.R_JOINT_PER_BOLT.magnitude.std_dev) * consts.ureg.microohm
    energy = (i_worst**2 * r_worst / n_eff * consts.FAULT_DURATION).to(consts.ureg.J)
    ratio = (energy / (1.9 * consts.ureg.MWh)).to(consts.ureg.dimensionless)
    print(f"{plate_name.upper()} fault energy: {energy:.1f} ; ratio to 1.9 MWh BESS: {ratio:.2e}")


CG fault energy: 5434.9 joule ; ratio to 1.9 MWh BESS: 7.95e-07 dimensionless
BG_AC fault energy: 297.0 joule ; ratio to 1.9 MWh BESS: 4.34e-08 dimensionless
BG_DC fault energy: 343.8 joule ; ratio to 1.9 MWh BESS: 5.03e-08 dimensionless
CD fault energy: 9.7 joule ; ratio to 1.9 MWh BESS: 1.42e-09 dimensionless


In [7]:
# Sanity 2: differential expansion order of magnitude.
for plate_name, (consts, _res) in results.items():
    delta_alpha = consts.PLATE_ALPHA - consts.FRAME_ALPHA
    per_meter = (delta_alpha * 1000 * consts.ureg.mm * 85 * consts.ureg.kelvin).to(consts.ureg.mm)
    print(f"{plate_name.upper()} differential expansion per meter at deltaT=85K: {per_meter:.3f}")


CG differential expansion per meter at deltaT=85K: 1.012 millimeter
BG_AC differential expansion per meter at deltaT=85K: 1.012 millimeter
BG_DC differential expansion per meter at deltaT=85K: 1.012 millimeter
CD differential expansion per meter at deltaT=85K: 1.012 millimeter
